In [ ]:
!python -m pip install horovod

import horovod.tensorflow.keras as hvd
import tensorflow as tf

# Inicializar Horovod
hvd.init()

# Configurar GPU local para este rank
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_visible_devices(gpus[hvd.local_rank()], 'GPU')

# Criar modelo
model = create_model()

# Otimizador distribuído
opt = keras.optimizers.Adam(learning_rate=0.001 * hvd.size())
opt = hvd.DistributedOptimizer(opt)

model.compile(optimizer=opt, loss='categorical_crossentropy')

# Treinar (cada rank processa uma parte dos dados)
model.fit(train_dataset, epochs=10, callbacks=[hvd.callbacks.BroadcastGlobalVariablesCallback(0)])

### Explicação do Treinamento Distribuído com Horovod

Este enta o treinamento distribuído usando o framework **Horovod** com Keras/TensorFlow. Aqui está o que acontece em cada etapa:

1.  **`hvd.init()`**: Inicializa o ambiente do Horovod. Isso permite que diferentes processos (ranks) se comuniquem.
2.  **Configuração de GPU**: O código detecta as GPUs disponíveis e atribui uma GPU específica para cada processo baseado no seu `local_rank`, evitando conflitos de memória.
3.  **Otimizador Distribuído**:
    *   Escalamos o `learning_rate` pelo número de processos (`hvd.size()`). Isso compensa o fato de estarmos processando mini-batches maiores de forma agregada.
    *   O `hvd.DistributedOptimizer` envolve o otimizador padrão para lidar com a média dos gradientes entre todos os trabalhadores (Allreduce).
4.  **`BroadcastGlobalVariablesCallback(0)`**: Garante que todos os processos comecem com os mesmos pesos iniciais, copiando o estado do processo 'mestre' (rank 0) para todos os outros no início do treinamento.